# Variant 04. Restaurant Performance Analysis

**Domain:** Restaurant Performance, Kyrgyzstan. All money is in KGS. Data are synthetic.
Tools used: NumPy, Pandas, Matplotlib only. The original CSV files are never modified; all cleaning is done on a copy.

## TASK 1. Initial Data Exploration

Both files start with a BOM character, so they are read with `encoding="utf-8-sig"`. Otherwise the first column would be called `\ufeffTip Amount`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw = pd.read_csv("04_restaurant_raw.csv", encoding="utf-8-sig")
lookup = pd.read_csv("04_restaurant_lookup.csv", encoding="utf-8-sig")
print("raw shape:", raw.shape, "| lookup shape:", lookup.shape)

raw shape: (840, 12) | lookup shape: (7, 3)


**1.1 Shape.** The main table has 840 rows and 12 columns; the lookup table has 7 rows and 3 columns.

**1.2 First 10 rows:**

In [2]:
raw.head(10)

,Tip Amount,Payment Method,Customer Count,Waiter,Unit Price,Quantity,dish_name_raw,Dish Category,Meal Type,Order Date,Branch,Order ID
0,No,Basic,No,Premium,19094,38,Yes,Category A,Standard,2025-06-15,Bishkek,04_-00000
1,Yes,Basic,No,Premium,41266,21,Yes,Category B,Premium,28.11.2024,Bishkek,04_-00001
2,No,Basic,No,Premium,48267,16,Yes,Category A,Premium,2024-12-08,Bishkek,04_-00002
3,Yes,Premium,Yes,Basic,25182,49,No,Category C,Basic,2024-02-09,Tokmok,04_-00003
4,No,Basic,Yes,Standard,22174,39,No,Category C,Basic,23.12.2025,Bishkek,04_-00004
5,Yes,Premium,Yes,Premium,30596,41,No,Category A,Basic,16.05.2024,Bishkek,04_-00005
6,Yes,Standard,No,Standard,4950,14,No,Category C,Basic,2025-10-14,Bishkek,04_-00006
7,No,Standard,No,Premium,39254,4,Yes,Category A,Premium,04.04.2025,Jalal-Abad,04_-00007
8,No,Standard,No,Basic,16132,25,Yes,Category A,Basic,2025-09-14,Karakol,04_-00008
9,Yes,Basic,No,Basic,49238,11,Yes,Category C,Basic,27.06.2024,Tokmok,04_-00009


**1.3 Column names and types (`.info()`).** Column names mix spaces, capitals and snake_case (`dish_name_raw`). `Order Date` is `object` (text), not a date. `Unit Price` and `Quantity` are already `int64` in this file, although the task describes `Quantity` as text; the `astype()` conversion is still applied in Task 2 so the notebook is safe if the text version is used.

In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Tip Amount      840 non-null    str  
 1   Payment Method  840 non-null    str  
 2   Customer Count  840 non-null    str  
 3   Waiter          840 non-null    str  
 4   Unit Price      840 non-null    int64
 5   Quantity        840 non-null    int64
 6   dish_name_raw   840 non-null    str  
 7   Dish Category   825 non-null    str  
 8   Meal Type       827 non-null    str  
 9   Order Date      840 non-null    str  
 10  Branch          840 non-null    str  
 11  Order ID        840 non-null    str  
dtypes: int64(2), str(10)
memory usage: 78.9 KB


**1.4 Unique values.** `Meal Type` has 11 spellings for 3 real categories (`Basic`, `basic`, ` Basic `, ...) plus missing values. `Payment Method` is clean: only `Basic`, `Premium`, `Standard`.

In [4]:
print(raw["Meal Type"].unique())
print(raw["Meal Type"].value_counts(dropna=False))
print(raw["Payment Method"].unique())
print(raw["Payment Method"].value_counts(dropna=False))

<StringArray>
[  'Standard',    'Premium',      'Basic',    'premium',          nan,
   'standard',    ' Basic ',  ' Premium ', ' Standard ',      'basic',
 ' standard ']
Length: 11, dtype: str
Meal Type
Premium       269
Basic         258
Standard      245
premium        13
NaN            13
standard       10
 Basic          8
 Standard       8
basic           8
 Premium        7
 standard       1
Name: count, dtype: int64
<StringArray>
['Basic', 'Premium', 'Standard']
Length: 3, dtype: str
Payment Method
Standard    292
Premium     275
Basic       273
Name: count, dtype: int64


**1.5 Missing values.** `Meal Type` has 13 missing and `Dish Category` has 15 missing. No other column has missing values (the "N/A" mentioned in the task notes does not occur in this file).

In [5]:
raw.isna().sum()

Tip Amount         0
Payment Method     0
Customer Count     0
Waiter             0
Unit Price         0
Quantity           0
dish_name_raw      0
Dish Category     15
Meal Type         13
Order Date         0
Branch             0
Order ID           0
dtype: int64

**1.6 Duplicates.** There are 19 fully identical rows (the same `Order ID` and all other fields), shown below. There are 20 repeated `Order ID`s, so one `Order ID` is reused by two *different* records. That one is not a copy and is kept.

In [6]:
print("duplicate rows:", raw.duplicated().sum())
print("duplicate Order IDs:", raw["Order ID"].duplicated().sum())
raw[raw.duplicated(keep=False)].sort_values("Order ID").head(6)

duplicate rows: 19
duplicate Order IDs: 20


,Tip Amount,Payment Method,Customer Count,Waiter,Unit Price,Quantity,dish_name_raw,Dish Category,Meal Type,Order Date,Branch,Order ID
28,Yes,Basic,Yes,Premium,19088,13,No,Category B,Premium,12.04.2025,Jalal-Abad,04_-00028
837,Yes,Basic,Yes,Premium,19088,13,No,Category B,Premium,12.04.2025,Jalal-Abad,04_-00028
831,No,Basic,No,Premium,32078,45,No,Category B,Standard,2025-05-31,Jalal-Abad,04_-00228
228,No,Basic,No,Premium,32078,45,No,Category B,Standard,2025-05-31,Jalal-Abad,04_-00228
835,Yes,Standard,No,Premium,4758,22,Yes,Category D,Premium,21.02.2024,Karakol,04_-00236
236,Yes,Standard,No,Premium,4758,22,Yes,Category D,Premium,21.02.2024,Karakol,04_-00236


**1.7 Variable types and data quality problems.**

- **Numerical:** `Unit Price`, `Quantity`.
- **Categorical:** `Order ID`, `Order Date` (text), `Branch`, `Meal Type`, `Dish Category`, `Payment Method`, `Waiter`, plus `Tip Amount`, `Customer Count`, `dish_name_raw`.

Problems found:

1. `Meal Type` inconsistent: `premium`, ` Basic `, ` standard ` (case and spaces). 13 values missing.
2. `Dish Category` has 15 missing values.
3. 19 exact duplicate rows.
4. `Unit Price` has extreme outliers: the max is 1,728,760 while the median is 26,475 (about 65 times larger).
5. `Order Date` is text in two formats: `2025-06-15` and `28.11.2024` (415 of 840 rows use dots).
6. Column names are inconvenient (spaces, capitals, mixed style) and the order is illogical (the ID is last).
7. **Misleading columns:** `Tip Amount`, `Customer Count` and `dish_name_raw` contain only `Yes`/`No`. They are not amounts or counts. `Payment Method` and `Waiter` contain `Basic/Premium/Standard`, not real payment types or names. This cannot be fixed; it affects Task 3.3 (see there).
8. The lookup table has duplicate keys that differ only by case (`Basic` and `basic`, with different segments) and one row with an empty key.